# Emissions → Warming Attribution

Translate each Carbon Major entity's cumulative emission record into their proportional contribution to observed global warming, with uncertainty bounds from the FaIR v2 calibrated ensemble.

**Method**: Proportional attribution via TCRE (Transient Climate Response to Cumulative Emissions). Warming scales approximately linearly with cumulative CO2 emissions, so each entity's warming contribution equals their share of global cumulative fossil CO2 × total FaIR-modelled warming.

**Attribution chain position**: *Cumulative Emissions → Atmospheric Forcing → Climate Signal*

**Outputs**
- `data/processed/entity_warming_contribution.parquet` — per-entity warming contribution (°C) with 5–95th percentile uncertainty
- `data/processed/fair_global_temperature.parquet` — FaIR ensemble temperature timeseries

**Reference**: Ekwurzel et al. (2017); FaIR v2.2 (Smith et al.); fair-calibrate v1.4 (Leach et al. 2024)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pooch
from pathlib import Path

from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

PROC = Path("../../data/processed")
FIGS = Path("../../outputs/figures")

# RCMIP historical runs to 2014; we extend with flat emissions to 2021
START_YEAR = 1750
END_YEAR   = 2021
# Attribution baseline: IPCC AR6 convention
BASELINE_SLICE = slice(1850, 1901)   # 1850–1900 pre-industrial baseline
ANALYSIS_YEAR  = 2020                # report warming as of this year

## 1. Load calibrated FaIR posteriors

841 posterior samples from fair-calibrate v1.4 (Leach et al. 2024, GMD). These were constrained against observed warming, ocean heat content, and CMIP6 models. The ensemble covers the IPCC AR6 assessed likely range for ECS (2.5–4°C) and TCR (1.2–2.4°C).

In [ ]:
POSTERIORS_URL  = "https://zenodo.org/records/10566813/files/calibrated_constrained_parameters.csv"
POSTERIORS_HASH = "76e2b9156ed26aa2730aa7023f1e40025a3637fe479df18e822120eff001848c"

posteriors_path = pooch.retrieve(
    POSTERIORS_URL,
    known_hash=f"sha256:{POSTERIORS_HASH}",
    fname="fair_calibrate_v1.4_posteriors.csv",
)
df_configs = pd.read_csv(posteriors_path, index_col=0)

print(f"Posterior ensemble: {len(df_configs)} configurations")
print(f"\nKey climate parameter ranges:")
clim_cols = [c for c in df_configs.columns if c.startswith("clim_")]
print(df_configs[clim_cols].describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean","5%","50%","95%"]].T.to_string())

## 2. Run FaIR historical ensemble

All 841 configs run simultaneously on the RCMIP historical emissions scenario. Runtime is typically 30–60 seconds.

In [ ]:
%%time

config_ids = df_configs.index.astype(str).tolist()

f = FAIR()
f.define_time(START_YEAR, END_YEAR, 1)
f.define_scenarios(["historical"])
f.define_configs(config_ids)

species, properties = read_properties()
f.define_species(species, properties)
f.allocate()

f.fill_from_rcmip()
f.fill_species_configs()

# Fill EBM climate configs from calibrated posteriors
fill(f.climate_configs["ocean_heat_capacity"],
     df_configs.loc[:, "clim_c1":"clim_c3"].values)
fill(f.climate_configs["ocean_heat_transfer"],
     df_configs.loc[:, "clim_kappa1":"clim_kappa3"].values)
fill(f.climate_configs["deep_ocean_efficacy"],
     df_configs["clim_epsilon"].values)
fill(f.climate_configs["forcing_4co2"],
     df_configs["clim_F_4xCO2"].values)
fill(f.climate_configs["gamma_autocorrelation"],
     df_configs["clim_gamma"].values)
fill(f.climate_configs["stochastic_run"], False)

initialise(f.concentration, 278.3, specie="CO2")
initialise(f.forcing, 0)
initialise(f.temperature, 0)
initialise(f.cumulative_emissions, 0)
initialise(f.airborne_emissions, 0)

f.run(progress=False)
print("FaIR run complete.")

In [ ]:
# Extract surface temperature (layer=0), relative to 1850–1900 baseline
t2m_raw = f.temperature.sel(scenario="historical", layer=0)  # (timebounds, configs)

baseline = t2m_raw.sel(timebounds=BASELINE_SLICE).mean("timebounds")
t2m = t2m_raw - baseline  # anomaly relative to 1850–1900

# Summary statistics across ensemble
t2m_median = t2m.median("config")
t2m_p05    = t2m.quantile(0.05, "config")
t2m_p95    = t2m.quantile(0.95, "config")

# Validation
mean_2011_2020 = float(t2m_median.sel(timebounds=slice(2011, 2021)).mean())
print(f"FaIR median warming 2011–2020 vs 1850–1900: {mean_2011_2020:.3f} °C")
print(f"IPCC AR6 best estimate:                      1.07 °C")
print()

p05_2020 = float(t2m_p05.sel(timebounds=ANALYSIS_YEAR))
p50_2020 = float(t2m_median.sel(timebounds=ANALYSIS_YEAR))
p95_2020 = float(t2m_p95.sel(timebounds=ANALYSIS_YEAR))
print(f"Total anthropogenic warming as of {ANALYSIS_YEAR}:")
print(f"  Median:  {p50_2020:.3f} °C")
print(f"  5–95th:  {p05_2020:.3f}–{p95_2020:.3f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
years = t2m_median.timebounds.values

ax.fill_between(years, t2m_p05.values, t2m_p95.values, alpha=0.25, color="#1976D2", label="5–95th percentile")
ax.plot(years, t2m_median.values, color="#1976D2", linewidth=2, label="FaIR median")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(1988, color="red",    linestyle="--", alpha=0.6, label="1988 (Hansen testimony)")
ax.axvline(2015, color="orange", linestyle="--", alpha=0.6, label="2015 (Paris Agreement)")
ax.set_title("FaIR ensemble: global mean surface temperature anomaly vs 1850–1900", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("°C anomaly")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fair_temperature_ensemble.png", bbox_inches="tight")
plt.show()

## 3. Global CO2 denominator

To convert entity emissions into a share of total global fossil CO2, we extract the cumulative CO2 FFI (fossil fuel & industry) total from the RCMIP emissions that FaIR used. This is the same dataset as the Global Carbon Project historical fossil CO2, ensuring consistency with the model run.

In [ ]:
# RCMIP CO2 FFI emissions — global fossil fuel CO2 (same for all configs)
co2_ffi_annual = f.emissions.sel(
    scenario="historical",
    config=config_ids[0],
    specie="CO2 FFI",
).to_series()
co2_ffi_annual.index = f.timebounds[:-1]  # emissions are mid-year

# Cumulative global fossil CO2 through ANALYSIS_YEAR (MtCO2)
global_cumul_co2_ffi = co2_ffi_annual.loc[:ANALYSIS_YEAR].sum() * 1000  # GtCO2 → MtCO2

print(f"Global cumulative CO2 FFI 1750–{ANALYSIS_YEAR}: {global_cumul_co2_ffi/1000:.1f} GtCO2")
print("(GCP 2023 reports ~450 GtC = ~1650 GtCO2 for fossil+cement 1850–2022)")

fig, ax = plt.subplots(figsize=(10, 4))
co2_ffi_annual.loc[1850:].plot(ax=ax)
ax.set_title("RCMIP global CO2 FFI emissions (fossil fuel & industry)")
ax.set_ylabel("GtCO₂ / year")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

## 4. Entity warming contributions

**Proportional attribution formula:**

$$\Delta T_{entity} = \frac{\text{cumulative entity CO}_2}{\text{cumulative global fossil CO}_2} \times \Delta T_{total}$$

This rests on TCRE — the approximately linear relationship between cumulative CO2 emissions and warming. Uncertainty in ΔT_total propagates directly to uncertainty in each entity's warming contribution.

**Note on scope**: Carbon Majors records cover both scope 3 (product combustion — ~88%) and scope 1 (operational). We compute contributions using `total_emissions_MtCO2e`. Scope 3 only and scope 1 only variants are also stored for sensitivity analysis.

In [ ]:
# Load Carbon Majors entity-year data
cm = pd.read_parquet(PROC / "cm_entity_year.parquet")
cm_to_2020 = cm[cm["year"] <= ANALYSIS_YEAR]

# Cumulative totals per entity (three scope variants)
entity_cumul = cm_to_2020.groupby(["parent_entity", "parent_type"]).agg(
    total_MtCO2e    = ("total_emissions_MtCO2e", "sum"),
    scope3_MtCO2    = ("product_emissions_MtCO2", "sum"),
    scope1_MtCO2e   = ("total_operational_emissions_MtCO2e", "sum"),
).reset_index()

# Carbon Majors total vs global fossil total
cm_total = entity_cumul["total_MtCO2e"].sum()
print(f"Carbon Majors cumulative total (to {ANALYSIS_YEAR}): {cm_total/1000:.1f} GtCO2e")
print(f"RCMIP global fossil CO2 (to {ANALYSIS_YEAR}):         {global_cumul_co2_ffi/1000:.1f} GtCO2")
print(f"Carbon Majors coverage: {cm_total/global_cumul_co2_ffi*100:.1f}% of global fossil CO2")

In [ ]:
# Entity share of GLOBAL cumulative fossil CO2
entity_cumul["global_share"] = entity_cumul["total_MtCO2e"] / global_cumul_co2_ffi
entity_cumul["global_share_scope3"] = entity_cumul["scope3_MtCO2"] / global_cumul_co2_ffi
entity_cumul["global_share_scope1"] = entity_cumul["scope1_MtCO2e"] / global_cumul_co2_ffi

# FaIR temperature at ANALYSIS_YEAR — one value per config
t_2020_ensemble = t2m.sel(timebounds=ANALYSIS_YEAR).values  # shape: (841,)

# Warming contribution = entity_share × ΔT (broadcast over ensemble)
shares = entity_cumul["global_share"].values[:, np.newaxis]        # (n_entities, 1)
warming_matrix = shares * t_2020_ensemble[np.newaxis, :]           # (n_entities, 841)

entity_cumul["warming_p05_degC"] = np.percentile(warming_matrix, 5,  axis=1)
entity_cumul["warming_p50_degC"] = np.percentile(warming_matrix, 50, axis=1)
entity_cumul["warming_p95_degC"] = np.percentile(warming_matrix, 95, axis=1)

# Sort by median warming contribution
entity_cumul = entity_cumul.sort_values("warming_p50_degC", ascending=False).reset_index(drop=True)

print(f"\nTop 15 entities by warming contribution as of {ANALYSIS_YEAR}:")
top15 = entity_cumul.head(15)[["parent_entity","parent_type","warming_p05_degC","warming_p50_degC","warming_p95_degC","global_share"]].copy()
top15["global_share"] = (top15["global_share"]*100).round(2).astype(str) + "%"
top15["warming"] = top15.apply(
    lambda r: f"{r.warming_p50_degC*1000:.1f} [{r.warming_p05_degC*1000:.1f}–{r.warming_p95_degC*1000:.1f}] m°C", axis=1
)
print(top15[["parent_entity","parent_type","global_share","warming"]].to_string(index=False))

In [ ]:
# Visualise: top 20 entity warming contributions with uncertainty
top20 = entity_cumul.head(20)
type_colors = {
    "Investor-owned Company": "#2196F3",
    "State-owned Entity":     "#FF5722",
    "Nation State":           "#4CAF50",
}

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top20))

colors = top20["parent_type"].map(type_colors)
ax.barh(y_pos, top20["warming_p50_degC"] * 1000,
        xerr=[
            (top20["warming_p50_degC"] - top20["warming_p05_degC"]) * 1000,
            (top20["warming_p95_degC"] - top20["warming_p50_degC"]) * 1000,
        ],
        color=colors, error_kw=dict(elinewidth=1, capsize=3), alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(top20["parent_entity"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel(f"Warming contribution as of {ANALYSIS_YEAR} (m°C, relative to 1850–1900)")
ax.set_title(f"Top 20 Carbon Majors — warming contribution (m°C)\nwith 5–95th percentile uncertainty from FaIR ensemble", fontsize=12)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "entity_warming_contributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scope sensitivity: how much does the scope 1 vs scope 1+3 choice matter?
shares_s3 = entity_cumul["global_share_scope3"].values[:, np.newaxis]
shares_s1 = entity_cumul["global_share_scope1"].values[:, np.newaxis]

entity_cumul["warming_scope3_p50"] = np.median(shares_s3 * t_2020_ensemble[np.newaxis, :], axis=1)
entity_cumul["warming_scope1_p50"] = np.median(shares_s1 * t_2020_ensemble[np.newaxis, :], axis=1)

print("Scope sensitivity — top 10 median warming contributions (m°C):")
sens = entity_cumul.head(10)[["parent_entity","warming_p50_degC","warming_scope3_p50","warming_scope1_p50"]].copy()
for col in ["warming_p50_degC","warming_scope3_p50","warming_scope1_p50"]:
    sens[col] = (sens[col] * 1000).round(1)
sens.columns = ["Entity", "Total (S1+S3)", "S3 only", "S1 only"]
print(sens.to_string(index=False))

## 5. Total Carbon Majors warming share

How much of total observed warming is attributable to the ~178 named entities in Carbon Majors collectively?

In [ ]:
cm_total_share = entity_cumul["global_share"].sum()
cm_warming_ensemble = cm_total_share * t_2020_ensemble

print("Carbon Majors collective warming contribution:")
print(f"  Share of global fossil CO2: {cm_total_share*100:.1f}%")
print(f"  Attributed warming (median): {np.median(cm_warming_ensemble):.3f} °C")
print(f"  5–95th percentile:           {np.percentile(cm_warming_ensemble,5):.3f}–{np.percentile(cm_warming_ensemble,95):.3f} °C")
print(f"  Total FaIR warming (median): {np.median(t_2020_ensemble):.3f} °C")
print()
print(f"  → Named Carbon Majors responsible for ~{np.median(cm_warming_ensemble)/np.median(t_2020_ensemble)*100:.0f}%",
      f"of total anthropogenic warming to {ANALYSIS_YEAR}")

## 6. Warming timeseries by entity type

Show how each entity type's accumulated warming contribution grew over time.

In [ ]:
# For each year, compute entity type cumulative share × FaIR T at that year
# Use the config whose clim_F_4xCO2 is closest to the ensemble median
median_label = str(df_configs["clim_F_4xCO2"].sub(df_configs["clim_F_4xCO2"].median()).abs().idxmin())
t_median_ts = t2m.sel(config=median_label).values

# Entity type shares by year (cumulative)
cm_raw = pd.read_parquet(PROC / "cm_entity_year.parquet")
type_annual = (
    cm_raw[cm_raw["year"] <= ANALYSIS_YEAR]
    .groupby(["year", "parent_type"])["total_emissions_MtCO2e"]
    .sum()
    .unstack("parent_type")
    .fillna(0)
    .sort_index()
)
type_cumul_share = type_annual.cumsum() / global_cumul_co2_ffi

fair_years = f.timebounds[:-1]
t_by_year = pd.Series(t_median_ts[:-1], index=fair_years.astype(int))

common_years = type_cumul_share.index.intersection(t_by_year.index)
type_warming = type_cumul_share.loc[common_years].multiply(t_by_year.loc[common_years], axis=0) * 1000  # m°C

# Plot from 1900 onward (pre-industrial attribution is negligible and can go negative vs baseline)
plot_data = type_warming.loc[1900:].clip(lower=0)

fig, ax = plt.subplots(figsize=(12, 5))
plot_data.plot.area(ax=ax, alpha=0.8)
ax.set_title("Warming attributable to Carbon Majors by entity type (median config)", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Attributed warming (m°C vs 1850–1900)")
ax.legend(loc="upper left", fontsize=9)
ax.axvline(1988, color="red",    linestyle="--", alpha=0.5)
ax.axvline(2015, color="orange", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGS / "entity_type_warming_timeseries.png", bbox_inches="tight")
plt.show()

## 7. Save outputs

In [ ]:
# Entity warming contribution table
entity_cumul.to_parquet(PROC / "entity_warming_contribution.parquet", index=False)

# FaIR temperature ensemble — store median + percentiles as a tidy DataFrame
fair_t = pd.DataFrame({
    "year":    f.timebounds.astype(int),
    "t_p05":   t2m_p05.values,
    "t_p50":   t2m_median.values,
    "t_p95":   t2m_p95.values,
})
fair_t.to_parquet(PROC / "fair_global_temperature.parquet", index=False)

print("Saved:")
print(f"  entity_warming_contribution.parquet  {len(entity_cumul):>4} rows")
print(f"  fair_global_temperature.parquet      {len(fair_t):>4} rows")
print()
print(f"Top 5 warming contributors (median, m°C vs 1850–1900 baseline, as of {ANALYSIS_YEAR}):")
for _, row in entity_cumul.head(5).iterrows():
    print(f"  {row.parent_entity:<45} {row.warming_p50_degC*1000:6.1f} [{row.warming_p05_degC*1000:.1f}–{row.warming_p95_degC*1000:.1f}]")

## Key findings

- **Total FaIR warming by 2020**: **1.18 °C** [5–95th: 0.87–1.57 °C] vs 1850–1900 (median = 1.074°C for 2011–2020, matching IPCC AR6 best estimate of 1.07°C ✓)
- **Carbon Majors collective**: **75.5%** of global fossil CO₂ → **0.89 °C** attributed warming [0.66–1.19] (~76% of total). Close to the ~71% Heede coverage figure; the small excess reflects the known CO₂e-numerator vs CO₂-FFI-denominator mismatch (see next steps).
- **Top entity**: Former Soviet Union (1900–1991) — **92.8 m°C** [68.4–123.5] (historical; Saudi Aramco is top active entity at 44.7 m°C)
- **Scope sensitivity**: S1+S3 total is ~**11×** larger than S1-only (e.g. Aramco: 44.7 vs 3.9 m°C); if liability frameworks exclude scope 3, per-entity figures shrink ~9–11×

> **Data-integrity note (2026-06-17):** an earlier version of this finding read 44.6% → 0.53 °C. That was an artifact of the `cm_entity_year.parquet` LEI-dropna bug, which silently discarded ~562 GtCO₂e of null-LEI emitters (Former Soviet Union, China Coal, Chevron, NIOC…). After the fix the collective share rises to 75.5%. Incumbent entities with valid LEIs (Saudi Aramco, ExxonMobil) are unchanged; the previously-dropped emitters now appear in the rankings. See `wiki/findings/2026-06-17-lei-dropna-fix.md`.

Top 5 entity warming contributions (m°C, p50 [p05–p95]):

| Entity | m°C |
|--------|-----|
| Former Soviet Union (1900–1991) | 92.8 [68.4–123.5] |
| China (Coal, 1945–2004) | 72.0 [53.1–95.9] |
| Saudi Aramco | 44.7 [32.9–59.5] |
| Chevron | 41.3 [30.5–55.0] |
| ExxonMobil | 37.6 [27.7–50.1] |

→ See `wiki/findings/2026-05-15-emissions-to-warming.md` for the full write-up.